# Création du CLIP

In [ ]:
!git clone https://github.com/LeoQUENETTE/Projet-ML2.git
%cd Projet-ML2
!git checkout clip
!git pull

Cloning into 'Projet-ML2'...
remote: Enumerating objects: 2160, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 2160 (delta 14), reused 19 (delta 9), pack-reused 2134 (from 2)
Receiving objects: 100% (2160/2160), 171.84 MiB | 38.27 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/Projet-ML2/Projet-ML2/Projet-ML2/Projet-ML2/Projet-ML2/Projet-ML2/Projet-ML2/Projet-ML2
Branch 'clip' set up to track remote branch 'clip' from 'origin'.
Switched to a new branch 'clip'
Already up to date.


In [ ]:
import sys
sys.path.append("src/classes")

In [ ]:
import os
import pandas as pd
import re
import numpy as np
import random
import zipfile
import requests
import io
import math
from pathlib import Path
import cv2

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.utils import register_keras_serializable, to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import load_model
from tensorflow.keras.metrics import Mean
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model
from tensorflow.keras.models import load_model


from src.classes.UsefullClasses import *
from src.classes.SmallBert import *
from src.classes.SmallBertClassification import *

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Pour utiliser au mieux le GPU
AUTOTUNE = tf.data.AUTOTUNE

## Récupération des architectures et poids des modèles de classification de test et d'image

In [ ]:
%ls src/models_forclip

best_image_classif.keras  best_smallbert.keras  best_smallbert_modified.keras


In [ ]:


saved_models_path = "./src/models_forclip/"

model_images = load_model(os.path.join(saved_models_path, "best_image_classif.keras"))

print("Modèle Image")
model_images.summary()

model_text = load_model(os.path.join(saved_models_path, "best_smallbert_textes.keras"),
              custom_objects={"PositionalEmbedding":PositionalEmbedding,
                              "TransformerBlock":TransformerBlock,
                              "SmallBERT":SmallBERT,
                              "SmallBERTForClassification":SmallBERTForClassification
                              })
print("Modèle Texte")
model_text.summary()

Modèle Image


Model: "Classif_Images_for_Clip"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input Layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_1 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_1 (MaxPooling2D)   │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_2 (Conv2D)               │ (None, 109, 109, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_2 (MaxPooling2D)   │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_3 (Conv2D)               │ (None, 52, 52, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_3 (MaxPooling2D)   │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_4 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_4 (MaxPooling2D)   │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_5 (Conv2D)               │ (None, 10, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_5 (MaxPooling2D)   │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_6 (Conv2D)               │ (None, 3, 3, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_6 (MaxPooling2D)   │ (None, 1, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense (Dense)                   │ (None, 100)            │        12,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 4)              │           404 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 900,938 (3.44 MB)

 Trainable params: 300,312 (1.15 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 600,626 (2.29 MB)

ValueError: Unrecognized keyword arguments passed to SmallBERTForClassification: {'dropout_rate': 0.2}

### Élagage et reconstruction des 2 modèles

In [ ]:
learning_rate=1e-3
loss_images="categorical_crossentropy"
loss_text="sparse_categorical_crossentropy"
metrics=["accuracy"]

cropped_model_images = model_images.layers[-4].output
new_model_images = Model(inputs=model_images.input, outputs=cropped_model_images)
new_model_images.compile(optimizer=Adam(learning_rate), loss=loss_images, metrics=metrics)

#cropped_model_text = model_text.layers[-2].output
#new_model_text = Model(inputs=model_text.input, outputs=cropped_model_text)
#new_model_text.compile(optimizer="adam", loss=loss_text, metrics=metrics)

In [ ]:
new_model_images.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input Layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_1 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_1 (MaxPooling2D)   │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_2 (Conv2D)               │ (None, 109, 109, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_2 (MaxPooling2D)   │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_3 (Conv2D)               │ (None, 52, 52, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_3 (MaxPooling2D)   │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_4 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_4 (MaxPooling2D)   │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_5 (Conv2D)               │ (None, 10, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_5 (MaxPooling2D)   │ (None, 5, 5, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_6 (Conv2D)               │ (None, 3, 3, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPooling2D_6 (MaxPooling2D)   │ (None, 1, 1, 128)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 287,008 (1.09 MB)

 Trainable params: 287,008 (1.09 MB)

 Non-trainable params: 0 (0.00 B)

###Normaliser les embeddings

In [ ]:
# Classe utile pour la partie Clip mais il fallait bien regarder pour la trouver
@register_keras_serializable()
class L2Normalize(layers.Layer):
    def __init__(self, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({"axis": self.axis})
        return config

###Loss

In [ ]:
# Perte contrastive CLIP
# Le but de cette fonction est d’aligner les embeddings d’images et de textes
# correspondants dans un espace latent partgé. Elle est inspirée du papier
# CLIP, où l'on entraîne le modèle à prédire quelle image correspond à quel
# texte et réciproquement.

@register_keras_serializable(package="clip")
class ClipLossLayer(layers.Layer):
    def __init__(self, temperature=0.07, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature
        self.clip_loss_metric = tf.keras.metrics.Mean(name="clip_loss")

    def call(self, inputs):
        # l'inputs est forcément (img, txt) ou [img, txt]
        img, txt = inputs  # (B, D) attention il faut avoir L2-normalisés !!

        # Matrice des similarités (cosinus parce qu'on a L2 zt ça simplifie)
        logits = tf.matmul(img, txt, transpose_b=True) / self.temperature

        # Les Labels implicites : c'est la diagonale
        labels = tf.range(tf.shape(logits)[0])

        li = tf.keras.losses.sparse_categorical_crossentropy(labels,
                                                             logits,
                                                             from_logits=True)
        lt = tf.keras.losses.sparse_categorical_crossentropy(labels,
                                                          tf.transpose(logits),
                                                          from_logits=True)
        loss = tf.reduce_mean(li + lt) / 2.0

        # Ca c'est super important car on ajoute la loss au graphe
        # du modèle et ça nous simplifie la vie
        # après on met à jour la métrique interne si on veut la suivre
        self.add_loss(loss)
        self.clip_loss_metric.update_state(loss)

        # On retourne un TUPLE de tenseurs(surtout pas une liste)
        # dc facileà récupérer
        return (img, txt)

    def get_config(self):
        return {**super().get_config(), "temperature": self.temperature}

In [ ]:
norm = L2Normalize()

image_embedding = new_model_images(new_model_images.input)
text_embedding = new_model_text(new_model_text.input)
img_emb = norm(image_embedding)
txt_emb = norm(text_embedding)
clip_img, clip_txt = ClipLossLayer([img_emb, txt_emb])

In [ ]:
@tf.keras.utils.register_keras_serializable()
class ClipModel(keras.Model):
    def __init__(self, inputs ,**kwargs):
        super().__init__(**kwargs)
        self.norm = L2Normalize()
        img_emb = norm(img_emb)
        txt_emb = norm(txt_emb)
        self.clip_loss = ClipLossLayer()
    def call(self, inputs):
        x_normed = self.norm(inputs)
        x = self.clip_loss(x_normed)
        return x

###Model Clip

In [ ]:
clip_model = tf.keras.Model(inputs=[image_input, text_input],outputs=[img_emb, txt_emb])
clip_model.compile(optimizer=tf.keras.optimizers.Adam(1e-4)) #loss deja fait, learning rate a ajuster

###Entrainement modele CLIP

In [ ]:
captions_csv_path = os.path.join(dataset_dir, "captions.csv")
def make_clip_dataset_smallbert(
    captions_csv_path,
    tokenizer_layer,
    batch_size=32,
    shuffle=True,
    drop_remainder=True,
    cache=True,
    seed=42,
):
    """
    Construit un tf.data.Dataset avec en sortie (images, token_ids) pour CLIP.
    """
    # On récupère le fichier captions.csv qui a tout
    df = pd.read_csv(captions_csv_path)
    image_paths = df["image_path"].astype(str).tolist()
    captions    = df["caption"].fillna("").astype(str).tolist()

    # Récupération du répertoire des images
    root = Path(dataset_dir)
    full_paths = [str(root / p) for p in image_paths]

    # Création du dataset d'image
    ds = tf.data.Dataset.from_tensor_slices((full_paths, captions))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(full_paths), seed=seed,
                        reshuffle_each_iteration=True)

    # Chargement d'une ensemble d'images normalisées et de tokens (le texte)
    IMAGE_H, IMAGE_W = 224, 224  # même que image_size

    def load_sample(img_path, caption):
        img = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [224, 224])
        img = tf.image.convert_image_dtype(img, tf.float32)  # [0,1]
        tokens = tf.cast(tokenizer_layer(caption), tf.int32)  # (L,)
        # x = dict des 2 entrées, pour forcer à ne pas avoir de  y
        return {"image_input": img, "text_input": tokens}

    # Pour utiliser le cache et pouvoir faire les traitements en //
    ds = ds.map(load_sample, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    # si drop_remainder=True on vire le dernier batch
    # s'il n'est pas de la bonne taille
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(AUTOTUNE)
    return ds

# Quanq le tokenizer est initialisé, un exemple d'appel que vous pourrez faire :
#train_dataset = make_clip_dataset_smallbert(captions_csv_path,
#                                       text_tokenizer,
#                                       batch_size=64)

In [ ]:
train_dataset = make_clip_dataset_smallbert(captions_csv_path,
                                       text_tokenizer, #verifier le nom
                                       batch_size=64)
model_history = clip_model.fit(
    train_dataset,
    epochs=10,
    verbose=1
)
clip_model.save("myclip_with_smallbert.keras")

### Faire de l'inférence

In [ ]:
# Charger le modèle
model_name="myclip_with_smallbert.keras"
clip = tf.keras.models.load_model(os.path.join(model_dir, model_name))

# ICI VOUS DEVEZ RECUPERER LA PARTIE ENCODEUR D'IMAGE
# ATTENTION SI VOUS VOULEZ PREDIRE AVEC CET ENCODEUR IL FAUT EN FAIRE UN MODELE
image_encoder_clip = tf.keras.Model(inputs=clip_model.input[0], outputs=clip_model.output[0])

# Lister toutes les images .jpg de la galerie
gallery_paths = [str(p) for p in Path(image_dir).rglob("*.jpg")]

# Normaliser les images et les mettre à la bonne taille
def load_img_224(p):
    x = tf.io.read_file(p)
    x = tf.image.decode_jpeg(x, channels=3)
    x = tf.image.convert_image_dtype(x, tf.float32)  # [0,1]
    x.set_shape([224, 224, 3])
    return x

# Création du dataset de manière efficace car utilisation du parallélisme
ds_gallery = (tf.data.Dataset.from_tensor_slices(gallery_paths)
              .map(load_img_224, num_parallel_calls=AUTOTUNE)
              .batch(64).prefetch(AUTOTUNE))

# C'est à partir de ds_gallery que l'on va obtenir les embeddings
# rappel en sortie ça doit être N = nb d'images et D la taille de
# l'embeddings appris par le modèle

# A COMPLETER gallery_embeds = ....
#gallery_embeds = image_encoder_clip.get_text_features(ds_gallery) ???
#pas compris a completer


# Sauvegarde compressée
index_path = os.path.join(model_dir, "image_index.npz")
np.savez_compressed(
    index_path,
    embeds=gallery_embeds.astype("float32"),
    paths=np.array(gallery_paths, dtype=np.str_)
)
print("Index enregistré à :", index_path,
      "- shape des embeddings :", gallery_embeds.shape)